# Simulación del dogma central de la biología molecular

**Autores:** Jaime Ercilla Martín, Javier Bolivar Garcia-Izquierdo

Este notebook sigue el flujo de la información genética desde el ADN hasta una proteína, usando como ejemplo el **gen de la insulina humana**:

```
ADN ──replicación──▶ ADN ──transcripción──▶ ARNm ──traducción──▶ proteína
```

Cada etapa tiene una celda de explicación (qué moléculas y enzimas intervienen) y una celda de código que ejecuta la simulación y muestra su resultado. El código de cada proceso está en los archivos `replicacion_dna.py`, `transcripcion.py`, `traduccion.py` y `utilidades.py`.

In [6]:
from utilidades import comp
from replicacion_dna import replicar
from transcripcion import transcribir
from traduccion import traducir

## 0. Datos de partida

La secuencia es el ARNm del gen de la **insulina** (*INS*, *Homo sapiens*), descargado de NCBI Gene (`rna.fna`). Está escrito con **T** en lugar de U, como si fuera ADN, y es la hebra que contiene la información del gen (hebra codificante).

Si el archivo trae varias variantes de transcrito, se usa solo la primera.

In [7]:
RUTA = "ncbi_dataset/data/rna.fna"

adn = ""
for linea in open(RUTA):
    if linea.startswith(">") and adn:      # solo la primera secuencia del FASTA
        break
    if not linea.startswith(">"):
        adn += linea.strip().upper()

assert adn and set(adn) <= set("ACGT"), "La secuencia debe contener solo A, C, G y T"
print(f"ADN de partida: {len(adn)} pb")

ADN de partida: 465 pb


## 1. Replicación del ADN (ADN → ADN)

La replicación copia la molécula completa antes de la división celular. Es **semiconservativa**: cada molécula hija conserva una hebra parental y lleva una hebra nueva.

| Enzima / molécula | Función en la simulación |
|---|---|
| **Topoisomerasa** | Alivia la tensión de la hélice por delante de la horquilla de replicación. |
| **Helicasa** | Rompe los puentes de hidrógeno entre bases y abre la doble hélice. |
| **Proteínas SSB** | Se unen a las hebras separadas y evitan que se vuelvan a unir. |
| **Primasa** | Sintetiza los **cebadores de ARN**, que dan a la polimerasa un extremo 3' desde el que empezar. |
| **ADN polimerasa III** | Añade nucleótidos en sentido 5'→3' complementarios al molde. |
| **ADN polimerasa I** | Elimina los cebadores de ARN y rellena el hueco con ADN. |
| **ADN ligasa** | Sella los cortes entre fragmentos contiguos. |

**Cadena líder.** Usa como molde la hebra B (3'→5') y se sintetiza de forma **continua** en el mismo sentido en que avanza la horquilla. Solo necesita un cebador.

**Cadena rezagada.** Usa como molde la hebra A, que se lee en sentido contrario al avance de la horquilla. Por eso se sintetiza de forma **discontinua**, en **fragmentos de Okazaki**: cada uno con su cebador, sintetizado 5'→3' en sentido contrario a la horquilla. Después se eliminan los cebadores y la ligasa une los fragmentos.

> Se usa la nomenclatura de procariotas (*E. coli*), que es la más habitual en los textos. En eucariotas, como el ser humano, los equivalentes son la polimerasa α-primasa (inicio), las polimerasas ε y δ (alargamiento) y RNasa H / FEN1 (eliminación de cebadores). El esquema general es el mismo.

In [8]:
lider = replicar(adn)


1. REPLICACION
  Topoisomerasa y helicasa abren la doble helice; las SSB estabilizan las hebras
  Hebra A (5'->3')    AGCCCTCCAGGACAGGCTGCATCAGAAGAGGCCATCAAGCAGATCACTGTCCTTCTGCCA...
  Hebra B (3'->5')    TCGGGAGGTCCTGTCCGACGTAGTCTTCTCCGGTAGTTCGTCTAGTGACAGGAAGACGGT...
  Cadena lider: primasa pone un cebador y ADN pol III sintetiza de forma continua
  Lider (5'->3')      AGCCCTCCAGGACAGGCTGCATCAGAAGAGGCCATCAAGCAGATCACTGTCCTTCTGCCA...
  Cadena rezagada: 16 fragmentos de Okazaki (un cebador y ADN pol III por fragmento)
  Ej. fragmento 1: CUCUU[ARN]+CTGATGCAGCCTGTCCTGGAGGGCT[ADN]
  ADN pol I quita los cebadores y ADN ligasa une los fragmentos
  Rezagada (5'->3')   GCTGGTTCAAGGGCTTTATTCCATCTCTCTCGGTGCAGGAGGCGGCGGGTGTGGGGCTGC...
  Moleculas hijas (hebra parental + nueva) identicas al original: OK


**Qué se ve en la salida.**

- `Hebra A` y `Hebra B` son las dos hebras parentales, con B complementaria de A.
- `Lider` coincide con A, porque se sintetiza sobre B (sintetizar el complemento de B devuelve A).
- `Rezagada` es el complemento de A, reunido a partir de los fragmentos de Okazaki.
- La última línea comprueba que las dos moléculas hijas (A + rezagada, y B + líder) son idénticas al ADN original.

## 2. Transcripción (ADN → ARN)

La **ARN polimerasa** usa una de las dos hebras de ADN como **molde** y sintetiza el ARNm en sentido 5'→3', con estas reglas de complementariedad:

| Base del molde de ADN | Base del ARNm |
|---|---|
| A | U |
| T | A |
| G | C |
| C | G |

El ARNm resultante tiene la misma secuencia que la hebra codificante, con **U en lugar de T**. Aquí la hebra codificante es la cadena líder obtenida en la replicación, y el molde es su complementaria.

In [9]:
arnm = transcribir(comp(lider))


2. TRANSCRIPCION
  ARN polimerasa lee el molde 3'->5' y sintetiza el ARNm 5'->3' (A-U, T-A, G-C, C-G)
  Molde ADN (3'->5')  TCGGGAGGTCCTGTCCGACGTAGTCTTCTCCGGTAGTTCGTCTAGTGACAGGAAGACGGT...
  ARNm (5'->3')       AGCCCUCCAGGACAGGCUGCAUCAGAAGAGGCCAUCAAGCAGAUCACUGUCCUUCUGCCA...


## 3. Traducción (ARN → proteína)

El **ribosoma** se une al ARNm y lo lee en tripletes (**codones**). Cada codón especifica un aminoácido según el código genético, y un **ARNt** con el anticodón complementario aporta ese aminoácido.

- La lectura empieza en el primer codón **AUG**, que codifica metionina (Met, M).
- Se va añadiendo un aminoácido por cada codón.
- Termina en un codón de **parada** (UAA, UAG o UGA), que no tiene ARNt: un factor de liberación libera la proteína.

Antes del AUG hay una región 5' no traducida del ARNm, que se salta.

In [10]:
traducir(arnm)


3. TRADUCCION
  Ribosoma inicia en el AUG (posicion 59); los ARNt aportan los aminoacidos
  Codones:     AUG GCC CUG UGG AUG CGC CUC CUG ...
  Aminoacidos: M   A   L   W   M   R   L   L   ...
  Parada: UAG

  Proteina (110 aminoacidos):
  MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGERGFFYTPKTRREAED
  LQVGQVELGGGPGAGSLQPLALEGSLQKRGIVEQCCTSICSLYQLENYCN


## 4. Resumen del flujo

| Etapa | Entrada | Salida |
|---|---|---|
| Replicación | ADN | dos moléculas de ADN idénticas a la original |
| Transcripción | hebra molde de ADN | ARNm |
| Traducción | ARNm | secuencia de aminoácidos |

La proteína obtenida es la **preproinsulina humana (110 aminoácidos)**. En la célula no es todavía la hormona activa: se corta el péptido señal (24 aminoácidos) para dar la proinsulina (86), y se elimina el péptido C para dejar la insulina madura (cadenas A y B, 51 aminoácidos en total). Estas modificaciones ocurren después de la traducción y no forman parte del dogma central.

## Simplificaciones del modelo

- Los cebadores de ARN tienen 5 nt y son complementarios al molde. Los fragmentos de Okazaki tienen un tamaño fijo de 30 nt.
- La transcripción parte directamente de la hebra molde, sin promotor ni terminador.
- La secuencia de partida es un ARNm ya procesado, por lo que no se modelan intrones ni el procesamiento del ARNm.